## Module 4: Evaluation

Evaluation is the process of measuring how well a search, RAG, or agent system performs using objective metrics instead of manual testing. It allows us to compare different search methods, prompts, models, and parameter settings. The module follows an A → Q → A'* pattern, where an LLM generates questions from known answers to create a test dataset with expected outputs. Search evaluation checks whether the correct documents are retrieved, while RAG evaluation measures the quality of generated answers, and agent evaluation also considers tool usage. Rather than the online evaluation where the system collects feedback from real users in production, this module mainly focuses on offline evaluation using synthetic data as a starting point, and introduces metrics such as Hit Rate, Mean Reciprocal Rank (MRR), and LLM-as-a-judge to evaluate system performance.

### Part 1: Search Evaluation
This part creates a ground truth dataset and uses it to evaluate retrieval quality.

#### _Section 1: Generating Ground Truth Data_
To evaluate a search system, we need a ground truth dataset that links each query to its correct document. This dataset can be created by human annotators, collected from real user queries, or generated synthetically with an LLM. In this module, an LLM is used to generate several questions for each FAQ document, making the original document the known correct answer for those questions. This synthetic dataset is then used to test whether the search system successfully retrieves the relevant document.

In [4]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/ingest.py
!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-12 20:17:14--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-12 20:17:14 (36.3 MB/s) - ‘ingest.py’ saved [738/738]

--2026-07-12 20:17:14--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1

In [5]:
# load the FAQ data
from ingest import load_faq_data
documents = load_faq_data()

In [6]:
# generate only for llm-zoomcamp course, since full set would take too long
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

print(f"Total llm-zoomcamp pages: {len(documents_llm)}")

# set llm-zoomcamp documents as the main documents to use for evaluation
documents = documents_llm

doc = documents[0]
print(f"Document ID: {doc['id']}")
print(f"Question: {doc['question']}")
print(f"Answer: {doc['answer']}")

Total llm-zoomcamp pages: 113
Document ID: 74eb249bbf
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


Each document needs a stable, unique ID because it serves as the label in the ground truth dataset. Since questions are generated from a specific document, its ID identifies the correct answer, allowing search evaluation to verify whether the system retrieved the right document.

An LLM is used to generate questions for each document using structured output, which returns data in a predefined format instead of free-form text. By defining the expected structure with a Pydantic model, the generated questions are returned as a consistent list of strings that can be easily processed by code without manual text parsing.

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

# define instructions for the llm
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
# make sure .env file is loaded (working directory is important here)
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


Until now we called `responses.create` and read `response.output_text`. For structured output we switch to `responses.parse` and pass `text_format=Questions`, which tells the API to return our class instead of free text.

In [10]:
import json

# prepare the document as json
user_prompt = json.dumps(doc)

# create the messages
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

# call the model
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

result = response.output_parsed
print(result)

# see the list of questions generated
print(result.questions)

questions=['Is it still okay to join this course if I found it late?', 'Can I start the course now even though it already began?', 'If I join late, will I still be able to get a certificate?', 'Do I need to submit my project before submissions close to receive the certificate?', 'What’s the deadline for project submission if I want the course certificate?']
['Is it still okay to join this course if I found it late?', 'Can I start the course now even though it already began?', 'If I join late, will I still be able to get a certificate?', 'Do I need to submit my project before submissions close to receive the certificate?', 'What’s the deadline for project submission if I want the course certificate?']


In [11]:
# import the helper file including reusable written functions
from evaluation_utils import llm_structured

# use the function call on the same document
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late to join the course if I found it only now?', 'Can new students still sign up for the course after it has already started?', 'If I join late, do I still have a chance to get a certificate?', 'What do I need to do to qualify for a certificate if I start the course now?', 'Are project submissions still open for people who discovered the course recently?']


In [ ]:
# track token usage
usage.input_tokens, usage.output_tokens

(207, 93)

In [16]:
# use helper function to calculate the cost of the request
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.00015525, 'output_cost': 0.0004185, 'total_cost': 0.00057375}

Each record has two fields:
- question: the question generated by the LLM
- document: the ID of the FAQ document that should answer the question

The document field connects the generated question to the document that contains the answer. Later, when we evaluate search, we'll ask the search engine the generated question. Then we'll check if it retrieves the document with this ID.

In [17]:
# convert the generated questions into ground truth records
records = []
for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to join the course if I found it only now?',
  'document': '74eb249bbf'},
 {'question': 'Can new students still sign up for the course after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, do I still have a chance to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for a certificate if I start the course now?',
  'document': '74eb249bbf'},
 {'question': 'Are project submissions still open for people who discovered the course recently?',
  'document': '74eb249bbf'}]

#### _Section 2: Generating Ground Truth for All Documents_

In this section we will generate questions and convert them to ground truth for every document in the FAQ dataset. Import below libraries to use progress bar and save results as CSV.

`uv add tqdm pandas`

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error, so we will be using the helper function that retries multiple times.

In [18]:
from evaluation_utils import llm_structured_retry

# create a function to create groun truth records for a given document
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [19]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

# try for the first 5 documents
for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

This works, but it runs one LLM call after another. Running it for all documents this way would take too long. Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

In [20]:
# ThreadPoolExecutor submits one job per document, updates the progress bar when a job finishes, and collects the results
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

565

In [21]:
from evaluation_utils import calc_price

total_cost = 0.0

# calculate the total cost
for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.087585

In [22]:
# using the written helper function
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.087585

In [25]:
import pandas as pd

# create the dataframe for the ground truth records
df_ground_truth = pd.DataFrame(ground_truth)

# save it for later use in evaluation
df_ground_truth.to_csv("data/ground_truth-current.csv", index=False)

For the simplicity, we will use the same document generated in the workshop rather than the one generated here

In [27]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget -O data/ground_truth-new.csv {PREFIX}/04-evaluation/data/ground_truth-new.csv

--2026-07-12 21:28:40--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/data/ground_truth-new.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 38613 (38K) [text/plain]
Saving to: ‘data/ground_truth-new.csv’

data/ground_truth-n 100%[===================>]  37.71K  --.-KB/s    in 0.001s  

2026-07-12 21:28:41 (56.1 MB/s) - ‘data/ground_truth-new.csv’ saved [38613/38613]



#### _Section 3: Search Evaluation_

In this section we will evaluate how well our search retrieves the correct documents. For each question in our ground truth dataset, we run search. Then we check whether the results include the correct document.

In [21]:
# could be also done in a separate notebook!
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-current.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [22]:
from ingest import load_faq_data, build_index

# load the documents and build a minsearch index
documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [23]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [24]:
# start with a single record
q = ground_truth[0]
print(f"Question: {q['question']}")
print(f"ID: {q['document']}")

# run search for the selected question
doc_id = q["document"]
results = text_search(query=q["question"])

# compare the retrieved document IDs with the ground truth document ID
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

# turn comparison into a list of 1s and 0s, where 1 means the retrieved document is relevant (matches the ground truth) and 0 means it is not relevant
relevance = []
for d in results:
    relevance.append(int(d["id"] == doc_id))
relevance

Question: I just found this course — can I still sign up and start now?
ID: 74eb249bbf
74eb249bbf == 74eb249bbf: True
04919992b3 == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


[1, 0, 0, 0, 0]

In [25]:
# create a function to compute relevance for a given question
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [26]:
# get relevance for the first question in the ground truth
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course — can I still sign up and start now?


[1, 0, 0, 0, 0]

In [27]:
from tqdm.auto import tqdm

# get relevance for all the documents
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

# call it for the first 15 records
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

# check results
relevance_total_text

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [17]:
# make relevance search generic
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

# run on the same page
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [28]:
# run for all records
relevance_total_all = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

#### _Section 4: Search Evaluation Metrics_

In this section we will turn computed relevance lists for search results into metrics.

- **Hit Rate:** Measures the fraction of queries where the correct document appears anywhere in the results.

In [29]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

cnt = 0
for line in example:
    if 1 in line:
        cnt = cnt + 1

print(f"Number of relevant documents (hits): {cnt}")
print(f"Relevance score: {cnt / len(example):.3f}")
# 0.933

Number of relevant documents (hits): 14
Relevance score: 0.933


In [30]:
# hit rate as a function
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

print(f"Relevance score: {hit_rate(example):.3f}")

Relevance score: 0.933


- **Mean Reciprocal Rank (MRR):** Hit rate does not include the position while MRR includes position of the right document. For each query, the score is based on the rank of the first correct document. Position 1: score is 1.0, position 2: score is 0.5, position 3: score is 0.333, not found: score is 0

In [31]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

print(f"MRR score: {total_score}")
print(f"Average : {total_score / len(example)}")

MRR score: 12.333333333333332
Average : 0.8222222222222222


In [32]:
# mrr as a function
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

print(f"MRR score: {mrr(example):.3f}")
# 0.822

MRR score: 0.822


In [33]:
# combining metrics into a single function
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8513274336283185, 'mrr': 0.7273156342182887}

When interpreting search evaluation metrics, remember that the ground truth assumes only one correct document per query, even though multiple retrieved documents may also be relevant. Synthetic questions often resemble the original text, which can artificially inflate metrics like Hit Rate and MRR, so very high scores should be interpreted cautiously. The acceptable performance threshold depends on the application and how well the downstream LLM can compensate for imperfect retrieval. Overall, a high MRR indicates that relevant documents appear near the top of the search results, making it easier for the LLM to generate accurate responses.

#### _Section 5: Search Parameter Tuning_

In this section we will use the search metrics from the preious section in order to tune the search parameters.

In [34]:
# define a search with configurable boost
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

# evaluate several boost values
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8884955752212389, 'mrr': 0.7879056047197639}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9061946902654867, 'mrr': 0.7891445427728611}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8513274336283185, 'mrr': 0.7273156342182887}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8247787610619469, 'mrr': 0.6917404129793508}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7858407079646018, 'mrr': 0.6642477876106194}


In [35]:
# define a search with all configurable parameters
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

# do a grid search for the best combination of parameters
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

# sort by MRR
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.969912,0.877817
19,2.0,4.0,0.2,0.969912,0.877817
35,5.0,10.0,0.5,0.969912,0.877817
22,2.0,10.0,0.2,0.966372,0.876932
7,1.0,4.0,0.2,0.969912,0.876608
23,2.0,10.0,0.5,0.969912,0.876490
18,2.0,4.0,0.1,0.971681,0.876136
34,5.0,10.0,0.2,0.971681,0.875870
33,5.0,10.0,0.1,0.971681,0.875811
21,2.0,10.0,0.1,0.962832,0.875103


The best combination weights answer twice as heavily as question, with almost no weight on section. So the data says the opposite of where we started. The answer text matters more for retrieval than the question text. The intuition was wrong, and we'd never have known without measuring it. This is exactly why we evaluate instead of guess.

The first three rows have the same relative weights:
question : answer : section = 1 : 2 : 0.1

So we can use the smaller and easier-to-read values where the numbers are not unnecessarily large. Hit Rate tells us whether the correct document appears at all. MRR tells us whether it appears near the top. A document near the top is more likely to be used by the RAG prompt.

In [36]:
# define search with updated parameters
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Evaluation helps identify the best search parameters, such as field boosts, filters, and the number of retrieved results, based on objective performance rather than guesswork. For a small number of parameters, grid search is an effective approach, while larger parameter spaces benefit from more efficient methods like random search or Bayesian optimization. These optimization techniques reduce evaluation time by focusing on parameter combinations that are most likely to improve search performance.

The number of retrieved search results (top-K) affects both retrieval performance and RAG efficiency. Increasing top-K can improve Hit Rate by giving the system more opportunities to retrieve the correct document, but it also increases the amount of context sent to the LLM, raising costs and making it harder for the model to focus on the most relevant information. For short FAQ-style documents, retrieving 5 results provides a good balance between accuracy and efficiency.

### Part 2: RAG and Agent Evaluation
This part evaluates answer quality after retrieval. It also shows the basic idea of agent evaluation: save the final answer and the tool-call trajectory.

RAG evaluation measures the performance of the entire pipeline, including search, prompt design, and the LLM, since errors in any of these components can lead to poor answers. In this part, RAG responses are evaluated using an LLM-as-a-judge, while agent evaluation considers both the quality of the final answer and the sequence of tool calls used to produce it.

Since RAG and agent responses are generated in natural language, they may differ in wording from the original answers while still conveying the same meaning. Therefore, the module uses an LLM-as-a-judge, which compares the question, the original answer, and the generated answer to determine whether they are semantically equivalent. The judge also explains its reasoning, which generally leads to more accurate evaluations than providing only a simple pass/fail verdict.

#### _Section 1: Generating RAG Answers_

In this section we evaluate the full RAG pipeline. For each generated question, we run RAG and save the answer produced by the LLM. Later, we'll compare this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our questions came from FAQ records. For each question, we know which original answer it came from.

In [ ]:
# could be done in a new notebook!
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-current.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
from ingest import load_faq_data, build_index

# load the documents and build a minsearch index
documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)